# feral — basic factor & solve

`feral` is a pure-Rust sparse **symmetric indefinite** direct solver (LDLᵀ with Bunch–Kaufman pivoting) that reports a **certified inertia** count. This notebook covers the core workflow: build a matrix, factor it, read the inertia, solve, refine, and reuse the symbolic analysis.

In [ ]:
import numpy as np
import feral

print('feral', feral.__version__)

## Build a matrix

`feral` stores symmetric matrices by their **lower triangle** in CSC form. The easiest constructors are `from_dense` (reads the lower triangle of a dense array) and `from_triplet` (COO lower-triangle entries).

In [ ]:
A = feral.CscMatrix.from_dense(np.array([
    [4.0, 1.0, 0.0],
    [1.0, 3.0, 2.0],
    [0.0, 2.0, 5.0],
]))
print(A)              # n=3, nnz counts the lower triangle
print('n =', A.n, ' nnz =', A.nnz)

## Factor and read the inertia

`Solver.factor` returns `(status, inertia)`. The inertia `(n_pos, n_neg, n_zero)` is the count of positive / negative / zero eigenvalues — exact for non-singular matrices. This SPD matrix is `(3, 0, 0)`.

In [ ]:
solver = feral.Solver()
status, inertia = solver.factor(A)
print('status :', feral.FactorStatus(status).name)
print('inertia:', inertia)
print('factor nnz   :', solver.factor_nnz)
print('cond. (1-norm):', f'{solver.estimate_condition_1norm(A):.3e}')
assert status == feral.FactorStatus.SUCCESS
assert inertia == feral.Inertia(3, 0, 0)

## Solve and check the residual

`relative_residual(x, b)` returns `‖A·x − b‖∞ / ‖b‖∞`.

In [ ]:
b = np.array([1.0, 2.0, 3.0])
x = solver.solve(b)
print('x =', x)
res = A.relative_residual(x, b)
print(f'relative residual = {res:.3e}')
assert res < 1e-12

## Iterative refinement

`solve_refined` runs a few steps of iterative refinement against the original matrix, recovering near machine precision.

In [ ]:
x_ref = solver.solve_refined(A, b)
print(f'refined residual = {A.relative_residual(x_ref, b):.3e}')
assert A.relative_residual(x_ref, b) < 1e-13

## Reuse the symbolic analysis (`refactor`)

On the interior-point hot path the sparsity **pattern** is fixed and only the **values** change each iteration. `refactor` reuses the cached symbolic analysis, so `symbolic_call_count` stays at 1.

In [ ]:
A2 = feral.CscMatrix.from_dense(np.array([
    [5.0, 1.0, 0.0],
    [1.0, 4.0, 2.0],
    [0.0, 2.0, 6.0],
]))
status2, inertia2 = solver.refactor(A2)
print('refactor status :', feral.FactorStatus(status2).name)
print('inertia         :', inertia2)
print('symbolic_call_count:', solver.symbolic_call_count)
assert solver.symbolic_call_count == 1